# Text Summarisation Model

Here data is loaded, BART model is used to summarise news articles as well as translate them

## Pre-requisites
If you are using google colab or a notebook, please uncomment this and install the packages.

In [ ]:
# Uncomment and install if on notebook
# !pip install transformers
# !pip install datasets
# !pip install evaulate
# pip install nltk
# pip install scikit-learn
# pip install bert-score#

If you are using your local machine, please install the required packages above. Given pip and python are installed; you can run `pip install -r requirements.txt`

## 1. Preparing data

### 1.1 Clean text

As text summarisation models require alot of the data to be kept in, text pre-processing function is very minimal. Because of this, i chose Rejex to clean the data Rejex is usead as it does not require much and meets the job, SPACY and nltk is more advanced but uneccessary for this job hence I chose rejex

In [ ]:
import re
def pre_process_txt(text):
    """Pre-process text by removing unwanted characters and formatting."""
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # Remove non-text characters but keep punctuation marks
    text = re.sub(r"[^\w\s\.,!?;:'\"-]", " ", text)

    # Collapse extra spaces
    cleaned_text = re.sub(r"\s+", " ", text).strip()
    
    return cleaned_text

### 1.2 Reduce text length to fit models token input limit

This model is usin Bart

Open the files uploaded to be read. Also applying text pre-processing function.
Try and except in case it failes
BartTokenizer
https://huggingface.co/transformers/v3.0.2/model_doc/bart.html#bartmodel

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BartTokenizer

nltk.download('punkt')

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

def reduce_by_importance(text, max_tokens=1024):
    # sentence split
    sentences = nltk.sent_tokenize(text)

    # rank with TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(sentences)
    scores = tfidf.sum(axis=1).A1

    # sort: highest importance first
    ranked = [s for _, s in sorted(zip(scores, sentences), reverse=True)]

    # now add sentences until tokens reach limit
    selected = []
    total_tokens = 0

    for sent in ranked:
        tokens = len(tokenizer.encode(sent, add_special_tokens=False))
        if total_tokens + tokens <= max_tokens:
            selected.append(sent)
            total_tokens += tokens
        else:
            break

    return " ".join(selected)


## 1.3 Load dataset

In [ ]:
from pathlib import Path

def load_dataset(base_path):
    articles_dir = Path(base_path) / "Articles"
    summary_dir = Path(base_path) / "Summary"

    if not articles_dir.exists() or not summary_dir.exists():
        raise FileNotFoundError("Articles or Summary directory not found.")

    # Get all article files and sort them
    article_files = sorted(articles_dir.glob("*.txt"))

    dataset = []
    
    for article_path in article_files:
            # Get corresponding summary file
            file_id = article_path.stem  # e.g., "001" from "001.txt"
            summary_path = summary_dir / f"{file_id}.txt"

            # Skip if summary doesn't exist
            if not summary_path.exists():
                print(f"Warning: No summary found for {file_id}")
                continue

            # Try and except block to handle potential read errors
            try:
                with open(article_path, 'r', encoding='utf-8') as f:
                    article = f.read()
                    article = pre_process_txt(article)
                    article_length = len(article.split())
                    if article_length > 1024:
                         print(f"Article {file_id} exceeds 1024 tokens, reducing by importance.")
                         print(article)
                         article = reduce_by_importance(article, max_tokens=1024)
                         print(article)
                    article_length = len(article.split())
                         
                with open(summary_path, 'r', encoding='utf-8') as f:
                    summary = f.read()
                    summary = pre_process_txt(summary)
                    summary_length = len(summary.split())

                dataset.append({
                    'id': file_id,
                    'article': article,
                    'summary': summary,
                    'article_length': article_length,
                    'summary_length': summary_length,
                })

            except Exception as e:
                print(f"Error processing {file_id}: {e}")
    
    print(f"Loaded {len(dataset)} article-summary pairs")
    # Knowing the maximum lengths of articles can help inform model choices and parameters
    print(f"Max article length: {max(d['article_length'] for d in dataset)} tokens")
    # Calculate max and min lengths which will be inform parameters to use for model training
    print(f"Max summary length: {max(d['summary_length'] for d in dataset)} tokens")
    print(f"Min summary length: {min(d['summary_length'] for d in dataset)} tokens")
    return dataset


## 1.4 Load actual dataset
Load the dataset and store it in variable. Also make notes of relevant figures

**Please confirm BASE_PATH variable here!** i.e. {BASE_PATH}/Articles or {BASE_PATH}/Summary.

Some IDEs may require a `/` at the beginning as well.

In [ ]:

BASE_PATH="training_dataset"

data = load_dataset(BASE_PATH)

**Recore the minimum and maximum length of summary to inform the model later on.**

In [ ]:
MAX_LEN = 660
MIN_LEN = 40

Article length is not noted as the `facebook/bart-large` models, including variations like `bart-large-mnli` and `bart-large-cnn` (which I will be using), are designed to handle a maximum input sequence length of 1024 tokens. This is already alot, if your dataset has even more tokens than 1024, a different model may be needed or a change in dataset.


### 1.5 Organise cleaned data into distinct groups the model will use

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import random

# Shuffle for randomness
random.seed(42)  # For reproducibility
random.shuffle(data)

# Calculate split indices
total = len(data)
train_end = int(total * 0.8)  # 80% train
test_end = train_end + int(total * 0.1)  # 10% test
# Remaining 10% for validation

# Split the data
train_items = data[:train_end]
val_items = data[train_end:test_end]
test_items = data[test_end:]

# Create datasets
train_dataset = Dataset.from_dict({
    "article": [item['article'] for item in train_items],
    "summary": [item['summary'] for item in train_items],
})

val_dataset = Dataset.from_dict({
    "article": [item['article'] for item in val_items],
    "summary": [item['summary'] for item in val_items],
})

test_dataset = Dataset.from_dict({
    "article": [item['article'] for item in test_items],
    "summary": [item['summary'] for item in test_items],
})

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 2. Train data


In [ ]:
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_func(examples):
  inputs = examples["article"]
  targets = examples["summary"]
  model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")

  with tokenizer.as_target_tokenizer():
    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length")

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

tokenized_dataset = dataset_dict.map(preprocess_func, batched=True)

In [ ]:

training_args = TrainingArguments(
    output_dir="./bart-large-cnn-model",
    learning_rate=3e-5, 
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

trainer.train()
trainer.save_model("./bart-large-cnn-model")

# BART


In [ ]:
# To store generated and reference summaries
generated_summary = []
reference_summary = []


for item in dataset_dict["test"]:
    article = item["article"]
    summary = item["summary"]

    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=1024,
        truncation=True # WHAT THIS MEAN
    ).to(model.device)

    # Generate Summary

    summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=6, # this
    no_repeat_ngram_size=3, # this
    length_penalty=1.0,
    max_length=MAX_LEN,
    min_length=MIN_LEN,)
    # TODO: Experiment with generation parameters
    # no_repeat_ngram_size=2,  # Avoid repetition
    # temperature=0.8,  # Add some randomness
    # do_sample=False,  # Use greedy/beam search)
    # Decode the summary
    gen_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    generated_summary.append(gen_summary)
    reference_summary.append(summary)
    


In [ ]:
import evaluate
from evaluate import load

rouge = load("rouge")
results = rouge.compute(
    predictions=generated_summary,
    references=reference_summary
)

print(results)

{'rouge1': np.float64(0.4203752451937245), 'rouge2': np.float64(0.3683674563184648), 'rougeL': np.float64(0.3565987731410186), 'rougeLsum': np.float64(0.3575572787895837)}

## Evaluation with BERTscorer
Rouge score is not accurate at evaluating readable content as it assesses wether the generated summary matches the actual summary. BERTscorer checks if the mean the same thing which is much more accurate when it comes to assessing the generated text.

In [ ]:
from bert_score import BERTScorer

scorer = BERTScorer(lang='en')

P, R, F1 = scorer.score(generated_summary, reference_summary)

# Average the results across the dataset
print("Precision: {:.4f}".format(P.mean().item()))
print("Recall: {:.4f}".format(R.mean().item()))
print("F1: {:.4f}".format(F1.mean().item()))

Precision: 0.9294
Recall: 0.8594
F1: 0.8927